# Aether Stage 2 — Phase 1 tiny overfit (run only after Phase 0 passes)

Mandatory Phase 1 from `docs/stage2_spec.md`: memorize exactly 64 `karl4th/limmim` transcription examples with frozen Stage 1 AetherSpeech and frozen Qwen3-4B, training only the R1 Connector. Hard limit: 5,000 optimizer steps. This notebook is not the full 460-hour run.

Every run writes directly to `MyDrive/aether-v3/stage2/runYYMMDD-HHMMSS`.


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, subprocess, sys, logging, time
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
from pathlib import Path
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "Add HF_TOKEN in Colab Secrets and enable notebook access"

REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_DIR = "/content/aether-v3"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "stage2"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "stage2", "--single-branch", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "stage2"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/stage2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR, "huggingface_hub", "pytest", "ruff", "mypy"], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
# A rerun may follow a git update in the same kernel. Drop stale project modules.
for module_name in list(sys.modules):
    if module_name == "aether_v3" or module_name.startswith("aether_v3."):
        del sys.modules[module_name]
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 1. Configuration and private Stage 1 checkpoint

In [ ]:
CONFIG_PATH = "configs/stage2_train.yaml"
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

from aether_v3.config import load_config
from aether_v3.models.aether_speech import AetherSpeechEncoder
from aether_v3.training.stage2_utils import load_stage1_encoder

cfg = load_config(CONFIG_PATH)
assert torch.cuda.is_available(), "Stage 2 requires a GPU runtime"
if cfg.llm.dtype == "bfloat16" and not torch.cuda.is_bf16_supported():
    print("GPU has no native BF16 support; using float16")
    cfg.llm.dtype = "float16"
    cfg.stage2_train.amp_dtype = "float16"
print("GPU:", torch.cuda.get_device_name(0), "LLM dtype:", cfg.llm.dtype)
stage1_path = hf_hub_download(
    repo_id=cfg.stage2_train.stage1_repo_id,
    filename=cfg.stage2_train.stage1_filename,
    revision=cfg.stage2_train.stage1_revision,
    token=os.environ["HF_TOKEN"],
)
encoder = AetherSpeechEncoder(cfg.aether_speech)
checkpoint = load_stage1_encoder(stage1_path, encoder)
encoder.eval()
print("Stage 1 encoder loaded; checkpoint step:", checkpoint.get("step", "unknown"))

## 2. Build the fixed 64-example transcription cache

The train cache is also the evaluation cache because this phase tests memorization, not generalization. Progress is logged after every encoder batch.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from aether_v3.data.stage2_cache import build_limmim_stage2_shards
from aether_v3.training.stage2_utils import create_run_dir

run_dir = create_run_dir(cfg.stage2_train.drive_root)
cache_root = run_dir / "cache" / "overfit64"
print("PHASE 1 RUN:", run_dir, flush=True)
tokenizer = AutoTokenizer.from_pretrained(cfg.llm.model_id, revision=cfg.llm.revision)
rows = load_dataset("karl4th/limmim", split="train", streaming=True, token=os.environ["HF_TOKEN"])
build_limmim_stage2_shards(rows, encoder, tokenizer, cache_root, "overfit", max_examples=64)
print("64-example cache ready", flush=True)


## 3. Run the bounded tiny overfit

Evaluates at step 0 and the pre-registered gates, logs generated transcripts and WER/CER, and stops unconditionally at 5,000 steps.


In [ ]:
from aether_v3.models.aether_speech_llm import AetherSpeechLLM
from aether_v3.training.stage2_utils import load_stage1_encoder
from aether_v3.training.train_stage2 import run_stage2_training

print("Loading frozen Qwen3-4B...", flush=True)
model = AetherSpeechLLM(cfg.aether_speech, cfg.connector, cfg.llm, speech_frozen=True)
load_stage1_encoder(stage1_path, model.encoder)
print("Starting Phase 1 tiny overfit...", flush=True)
run_stage2_training(cfg, run_dir, cache_root, cache_root, model=model, tokenizer=tokenizer)
print("PHASE 1 FINISHED:", run_dir, flush=True)


## 4. Inspect the gated result

In [ ]:
import json
rows = [json.loads(line) for line in open(run_dir/"log.jsonl")]
print("last records:")
for row in rows[-10:]: print(row)
print("checkpoints:")
for path in sorted(run_dir.glob("*.pt")): print(path.name)
print("periodic:", len(list((run_dir/"periodic").glob("*.pt"))))